# Rung-1 comparison: sample, 1a, 1b

Tasks: fastapi_15588,fastapi_15589,rich_3882,rich_3470

Generated by scripts/make_kaggle_notebook.py. Attach competition data, the wheelhouse dataset and the 31B model; accelerator L4 ×4; Run All; stop the session when done.

In [ ]:
# 1. Install the competition wheelhouse BEFORE anything imports google.adk
#    (Kaggle preinstalls another google-adk; importing it first causes pydantic class clashes).
import glob, importlib, os, shutil, subprocess, sys
from pathlib import Path

os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
if not WHEELHOUSE_DIR.exists():
    WHEELHOUSE_DIR = Path(glob.glob('/kaggle/input/**/gemma-4-developer-agent-wheelhouse', recursive=True)[0])

for pth in glob.glob('/usr/local/lib/python*/*-packages/*cutlass*.pth'):
    try:
        os.unlink(pth)
    except OSError:
        pass

tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target = tmp_whl / (w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name)
    if not target.exists():
        os.symlink(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
assert wheels, f'No wheels found in {WHEELHOUSE_DIR}'
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()
print('Wheelhouse installation complete.')

In [ ]:
# 2. Write the submissions to evaluate
import re
DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
SUB_ROOT = WORKING_DIR / 'submissions'
TASK_IDS = ['fastapi_15588', 'fastapi_15589', 'rich_3882', 'rich_3470']
CONFIGS = ['sample', '1a', '1b']
EMBEDDED = {
 "1a": {
  "agent.yaml": "name: executor\ndescription: Single agent that localizes, edits, verifies and submits a fix on its own.\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  # Declared only so a call never crashes the task: the harness task message may advertise them,\n  # and an undeclared tool call ends the task and discards the patch (forum 745028).\n  - search_similar_code\n  - get_code_neighbors\n  - get_code_subgraph\ngenerate_content_config: !include configs/sampling.yaml\n",
  "configs/sampling.yaml": "# Google's recommended sampling for Gemma 4 (model card); thinking off.\ntemperature: 1.0\ntop_p: 0.95\ntop_k: 64\nmax_output_tokens: 2048\nthinking_config:\n  include_thoughts: false\n",
  "eval_config.yaml": "evaluation:\n  timeout_seconds: 240\n  max_tool_calls: 60\n  max_time_minutes: 5.0\n  max_turns: 90\n",
  "prompts/system.md": "You are an autonomous software engineer fixing one task in the repository at /workspace.\nNobody will answer questions. Never ask; decide from the task text, the code and the tests.\n\n## The task (verbatim)\n{problem_description}\n\n## Budget\nYou have about 5 minutes, and time is spent on the text you write.\n- Every reply: one short plain-text progress line, then exactly one tool call. No long analysis.\n- Target: locate within ~8 calls, first edit by ~call 15, submit by ~call 30.\n- get_status is free: check it when unsure. Under 75 seconds left: make your best edit and submit.\n\n## Workflow\n1. Notes: first call, write /tmp/notes.md with run_command: exact names, strings and values from the task\n   (quoted verbatim), expected vs actual behaviour, and \"done when\".\n2. Find the source package (the workspace listing you were given often hides it):\n   git ls-files '*.py' | grep -v -e tests -e docs | head -30\n3. Locate: git grep -n 'NAME' -- '*.py' | head -20.\n   Short or title-only task: git log --oneline -15, git log -S'SYMBOL' --oneline | head,\n   git show SHA --stat, and look at docs/, docs_src/ and similar existing features.\n   Prefer git grep; use the graph tools only with an exact function or class name.\n4. Read narrowly: read_file with start_line/end_line, at most 60 lines at a time.\n   Append key file:line facts to /tmp/notes.md; re-read notes instead of re-reading files.\n5. Reproduce if quick: /tmp/repro.py. Before editing, note which existing tests already fail.\n6. Edit: small edit_file changes in source files. Never edit tests, conftest.py, pytest.ini or config.\n7. Verify: re-run the repro and the nearest test file:\n   python -m pytest tests/test_x.py -x -q 2>&1 | tail -20\n8. Pre-submit: git status --short and git diff --stat \u2192 only intended source files, non-empty.\n9. Call submit_patch, then reply with one short sentence. Make no edits after submitting.\n\n## Rules\n- Keep outputs small: pipe long commands through | head -40 or | tail -20; never print whole files.\n  rg and tree are not installed; use git grep. No pip, no network.\n- Shell quoting: use single quotes inside commands (git grep -n 'class Foo'). Never write backslash-escaped quotes.\n- Never run the identical command twice. If it did not help, change it or move on.\n- Scratch files only in /tmp; anything created in /workspace ends up in the patch.\n- Git is read-only: log, show, grep, blame, diff, status. Never checkout, restore, reset, stash, clean,\n  rebase or commit.\n- Ambiguous task: choose the reading that matches the exact wording and existing code conventions,\n  and note it in /tmp/notes.md.\n- An imperfect edit beats no edit: the working tree is graded even if you run out of time.\n"
 },
 "1b": {
  "agent.yaml": "name: orchestrator\ndescription: Coordinates the fix; explores briefly, then delegates reading, editing and verifying to helpers.\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include prompts/orchestrator.md\ntools:\n  - run_command\n  - read_file\n  - get_status\n  - submit_patch\n  # Declared only so a call never crashes the task (forum 745028); the prompt prefers git grep.\n  - search_similar_code\n  - get_code_neighbors\n  - get_code_subgraph\n  # skip_summarization stays false: true would end the orchestrator's run after every helper call.\n  - agent_tool:\n      config_path: sub_agents/reader.yaml\n  - agent_tool:\n      config_path: sub_agents/executor.yaml\n  - agent_tool:\n      config_path: sub_agents/verifier.yaml\ngenerate_content_config: !include configs/sampling.yaml\n",
  "configs/sampling.yaml": "# Google's recommended sampling for Gemma 4 (model card); thinking off.\ntemperature: 1.0\ntop_p: 0.95\ntop_k: 64\nmax_output_tokens: 2048\nthinking_config:\n  include_thoughts: false\n",
  "eval_config.yaml": "evaluation:\n  timeout_seconds: 240\n  max_tool_calls: 80  # helper calls share this budget\n  max_time_minutes: 5.0\n  max_turns: 90\n",
  "prompts/executor.md": "You are the engineer who makes the code change for one task in the repository at /workspace.\nApply the change you were asked to make. Nobody will answer questions.\n\n## The task (verbatim)\n{problem_description}\n\n## How to work\n- Every reply: one short plain-text progress line, then exactly one tool call. Use at most 10 tool calls.\n- Read the target lines first (read_file with start_line/end_line), then make small edit_file changes.\n  Copy old_string exactly from what read_file showed.\n- Edit source files only. Never edit tests, conftest.py, pytest.ini or config files.\n- Scratch files only in /tmp; anything created in /workspace ends up in the patch.\n- Quick check after editing: python -c \"import PACKAGE\" or a short /tmp/repro.py, output piped through | tail -20.\n- Use single quotes inside shell commands (git grep -n 'class Foo'); never backslash-escaped quotes.\n- Never run the identical command twice. Git is read-only: never checkout, restore, reset, stash, clean or commit.\n\n## Final message\nWhen done, your last message must contain only this, nothing else:\nCHANGED: path/to/file.py: one line describing the change (one line per file)\nCHECK: the command you ran and its result in one line\n",
  "prompts/orchestrator.md": "You are the lead engineer coordinating the fix of one task in the repository at /workspace.\nYou never edit code yourself: the executor helper makes all changes. Nobody will answer questions.\nNever ask; decide from the task text, the code and the tests.\n\n## The task (verbatim)\n{problem_description}\n\n## Budget\nYou have about 5 minutes for everything, including the helpers' work, and time is spent on the text written.\n- Every reply: one short plain-text progress line, then exactly one tool call. No long analysis.\n- get_status is free. Under 75 seconds left: stop delegating, check git diff --stat, and submit.\n\n## Phase A: look around yourself (cheap, read-only)\n1. First call: write /tmp/notes.md with run_command: exact names, strings and values from the task\n   (quoted verbatim), expected vs actual behaviour, and \"done when\".\n2. Find the source package (the workspace listing you were given often hides it):\n   git ls-files '*.py' | grep -v -e tests -e docs | head -30\n3. Locate: git grep -n 'NAME' -- '*.py' | head -20. Short or title-only task: git log --oneline -15,\n   git log -S'SYMBOL' --oneline | head, and look at docs/, docs_src/ and similar existing features.\n   Prefer git grep; use the graph tools only with an exact function or class name.\n4. Read at most 60 lines at a time; append key file:line facts to /tmp/notes.md.\nCall get_status after each step. As soon as tool_calls_used is 6 or more, stop reading yourself and go to Phase B.\n\n## Phase B: delegate\nEach helper starts with an empty memory: it knows the task text, but nothing you found.\nEvery request must be self-contained: file paths with line numbers, exact names, what to do, what to return.\n1. reader: ask one concrete question, for example where a value is validated and what the fix should be.\n   Skip it if you already know the exact location and fix.\n2. executor: give it the file:lines, the relevant current code lines and the exact change to make.\n3. verifier: tell it what changed and which behaviour must now hold.\n4. If the verdict is FAIL: send the executor the verifier's EVIDENCE and FIX, then verify again.\n   At most 2 fix rounds.\n5. Before submitting: run_command git diff --stat must show changed source files. Then call submit_patch,\n   reply with one short sentence, and stop.\nSubmit after a PASS, after 2 fix rounds, or when under 75 seconds are left.\n\n## Helper replies\n- Expected formats: reader LOCATION / EVIDENCE / FIX PLAN; executor CHANGED / CHECK; verifier VERDICT / EVIDENCE / FIX.\n- If a reply is not in that format, use whatever useful facts it contains.\n- If a reply is empty, ask the same helper once more with a narrower request, then continue on your own.\n- Trust command output over opinions: a FAIL without real command output is not a reason to undo a change.\n\n## Rules\n- Keep outputs small: pipe long commands through | head -40 or | tail -20; never print whole files.\n  rg and tree are not installed; use git grep. No pip, no network.\n- Shell quoting: use single quotes inside commands (git grep -n 'class Foo'). Never write backslash-escaped quotes.\n- Never run the identical command twice. If it did not help, change it or move on.\n- Git is read-only: log, show, grep, blame, diff, status. Never checkout, restore, reset, stash, clean,\n  rebase or commit.\n- Ambiguous task: choose the reading that matches the exact wording and existing code conventions.\n- An imperfect change beats no change: the working tree is graded even if time runs out.\n",
  "prompts/reader.md": "You are a read-only code investigator helping fix one task in the repository at /workspace.\nAnswer the question you were given. Never edit or create files outside /tmp. Nobody will answer questions.\n\n## The task (verbatim)\n{problem_description}\n\n## How to work\n- Every reply: one short plain-text progress line, then exactly one tool call. Use at most 8 tool calls.\n- Locate with git grep -n 'NAME' -- '*.py' | head -20; the source package is listed by\n  git ls-files '*.py' | grep -v -e tests -e docs | head -30.\n- Read narrowly: read_file with start_line/end_line, at most 60 lines at a time.\n- Use single quotes inside shell commands (git grep -n 'class Foo'); never backslash-escaped quotes.\n- Keep outputs small (| head -40). Never run the identical command twice. Git is read-only.\n\n## Final message\nWhen done, your last message must contain only this, nothing else:\nLOCATION: path/to/file.py:START-END (one line per place that must change)\nEVIDENCE: the 15 most relevant code lines, copied verbatim with their line numbers\nFIX PLAN: one or two sentences saying exactly what to change\n",
  "prompts/verifier.md": "You are the verifier for a fix of one task in the repository at /workspace.\nCheck the current change with real commands. Never edit source files; write only to /tmp. Nobody will answer questions.\n\n## The task (verbatim)\n{problem_description}\n\n## How to work\n- Every reply: one short plain-text progress line, then exactly one tool call. Use at most 8 tool calls.\n1. git status --short and git diff --stat: only intended source files changed, no tests or config, no scratch files.\n2. git diff | head -60: does the change match the task's exact names, messages and values?\n3. Write /tmp/repro.py that exercises the required behaviour; run it: python /tmp/repro.py 2>&1 | tail -20\n4. Run the nearest existing test file: python -m pytest tests/test_x.py -x -q 2>&1 | tail -20\n   A failure that is unrelated to the change and to the task is pre-existing: ignore it.\n- Use single quotes inside shell commands (git grep -n 'class Foo'); never backslash-escaped quotes.\n- Never run the identical command twice. Git is read-only.\n\n## Final message\nWhen done, your last message must contain only this, nothing else:\nVERDICT: PASS or FAIL\nEVIDENCE: the command that decided it and at most 20 lines of its real output\nFIX: for FAIL, the one thing to change (file:line and what); for PASS, write none\n",
  "sub_agents/executor.yaml": "name: executor\ndescription: >-\n  Makes the code change. Give it the exact file:lines, the current code and the fix to apply.\n  Returns CHANGED and CHECK.\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include ../prompts/executor.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\ngenerate_content_config: !include ../configs/sampling.yaml\n",
  "sub_agents/reader.yaml": "name: reader\ndescription: >-\n  Read-only code investigator. Give it one concrete question (what to find, which names to look for).\n  Returns LOCATION, EVIDENCE (verbatim code lines) and FIX PLAN.\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include ../prompts/reader.md\ntools:\n  - run_command\n  - read_file\ngenerate_content_config: !include ../configs/sampling.yaml\n",
  "sub_agents/verifier.yaml": "name: verifier\ndescription: >-\n  Checks the current change by running a reproduction and the nearest tests. Give it what was changed and\n  what behaviour to check. Returns VERDICT (PASS or FAIL), EVIDENCE and FIX.\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include ../prompts/verifier.md\ntools:\n  - run_command\n  - read_file\ngenerate_content_config: !include ../configs/sampling.yaml\n"
 }
}

shutil.rmtree(SUB_ROOT, ignore_errors=True)
for name, files in EMBEDDED.items():
    for rel, text in files.items():
        path = SUB_ROOT / name / rel
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding='utf-8')

if 'sample' in CONFIGS:
    # Google's sample as shipped, minus its LoRA adapters.
    sample = SUB_ROOT / 'sample'
    shutil.copytree(DATA_DIR / 'sample_submission', sample)
    shutil.rmtree(sample / 'adapters', ignore_errors=True)
    for yml in list(sample.rglob('*.yaml')):
        yml.write_text(re.sub(r'^adapter:.*\n', '', yml.read_text(), flags=re.M))
    (sample / 'eval_config.yaml').write_text(
        'evaluation:\n  timeout_seconds: 240\n  max_tool_calls: 100\n  max_time_minutes: 5.0\n  max_turns: 90\n')

for name in CONFIGS:
    print(name, sorted(p.relative_to(SUB_ROOT / name).as_posix() for p in (SUB_ROOT / name).rglob('*') if p.is_file()))

In [ ]:
# 3. Start vLLM once for all configs (no adapters → LoRA off → more KV cache). Takes ~6–8 min.
import litellm, torch
from adk_submission import VllmConfig, VllmServer

litellm.drop_params = True
TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
MODEL_PATH = Path(glob.glob('/kaggle/input/models/google/gemma-4/**/gemma-4-31b-it-qat-w4a16-ct/2', recursive=True)[0])
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
)
server_instance = VllmServer(vllm_cfg)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')
models = server_instance.create_model_registry(aliases=[TARGET_MODEL_NAME], model_prefix='openai/', api_key='EMPTY')

In [ ]:
# 4. Evaluate every config on the same tasks (Evaluator configured like the scorer; see scripts/run_eval.py)
import asyncio, concurrent.futures, time, yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator

DATA = DATA_DIR  # default used by build_config (copied from scripts/run_eval.py)


def read_eval_config(submission_dir):
    """The four eval_config.yaml keys the scorer reads (defaults as in the host notebook)."""
    path = Path(submission_dir) / "eval_config.yaml"
    raw = yaml.safe_load(path.read_text()) if path.exists() else {}
    section = (raw or {}).get("evaluation", raw or {})
    return {
        "timeout_seconds": int(section.get("timeout_seconds", 300)),
        "max_tool_calls": section.get("max_tool_calls"),
        "max_time_minutes": float(section.get("max_time_minutes", 5.0)),
        "max_turns": section.get("max_turns"),
    }


def wheels_dir_for(data_dir):
    """Official test wheels, plus data/wheels_extra if present (local only: test deps missing from the public set,
    forum 745775). Merged as links into data/wheels_merged so the official folder stays untouched."""
    official, extra = Path(data_dir) / "wheels", Path(data_dir) / "wheels_extra"
    if not extra.is_dir():
        return official
    merged = Path(data_dir) / "wheels_merged"
    merged.mkdir(exist_ok=True)
    for wheel in [*official.glob("*.whl"), *extra.glob("*.whl")]:
        link = merged / wheel.name
        if not link.exists():
            link.hardlink_to(wheel)
    return merged


def build_config(submission_dir, task_ids, results_dir, models, data_dir=DATA, sandbox="docker",
                 adapter_manifest=None, display_mode="single"):
    limits, gen_constraints = build_submission_limits()
    budgets = read_eval_config(submission_dir)
    return EvalConfig(
        tasks_path=Path(data_dir) / "tasks.jsonl",
        snapshots_dir=Path(data_dir) / "snapshots",
        results_dir=Path(results_dir),
        submission_dir=Path(submission_dir),
        models=models,
        sandbox=sandbox,
        wheels_dir=wheels_dir_for(data_dir),
        graph_dir=str(Path(data_dir) / "graphs"),
        embeddings_dir=str(Path(data_dir) / "embeddings"),
        task_ids=list(task_ids),
        limits=limits,
        generation_constraints=gen_constraints,
        adapter_manifest=adapter_manifest,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(
            compaction_interval=5, overlap_size=2, token_threshold=14336, event_retention_size=5,
        ),
        display_mode=display_mode,
        **budgets,
    )


def run_sync(coro):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(coro)).result()
    return asyncio.run(coro)


RESULTS = WORKING_DIR / 'results'
shutil.rmtree(RESULTS, ignore_errors=True)
wall = {}
for name in CONFIGS:
    start = time.time()
    config = build_config(SUB_ROOT / name, TASK_IDS, RESULTS / name, models, data_dir=DATA_DIR,
                          sandbox='subprocess', display_mode='single')
    result = run_sync(Evaluator(config).run())
    wall[name] = time.time() - start
    print(f'==> {name}: {result.resolved}/{result.total} resolved in {wall[name] / 60:.1f} min')

In [ ]:
# 5. Comparison table + results.zip (download it from the Output panel), then STOP THE SESSION.
import json
from collections import Counter


NUDGE_PREFIXES = ("Please continue your work", "Your previous response reached the token limit")


HELPER_FORMATS = {"reader": "LOCATION:", "executor": "CHANGED:", "verifier": "VERDICT:"}


TEST_PATHS = ("tests/", "test_", "conftest.py", "pytest.ini")


def bucket(task):
    """Coarse failure bucket from the harness result."""
    error = (task.get("error") or "").lower()
    patch = task.get("agent_patch") or ""
    if task.get("resolved"):
        return "resolved"
    if "context" in error and ("window" in error or "length" in error):
        return "overflow"
    if "not found" in error and "tool" in error:
        return "unknown tool"
    if not patch.strip():
        return "timeout, no patch" if "timeout" in error or "timed out" in error else "no patch"
    changed = [line.split(" b/")[-1] for line in patch.splitlines() if line.startswith("diff --git")]
    if any(any(t in f for t in TEST_PATHS) for f in changed):
        return "edited tests"
    return "wrong fix"


def trace_stats(trace):
    steps = trace.get("steps", [])
    stats = Counter()
    tools = Counter()
    commands = Counter()
    helper_ok = Counter()
    helper_total = Counter()
    last_prompt = {}  # per agent: helpers have their own conversations
    peak_prompt = 0
    for step in steps:
        message = str(step.get("message") or "")
        if step.get("source") != "agent" and message.startswith(NUDGE_PREFIXES):
            stats["nudges"] += 1
        metrics = step.get("metrics") or {}
        prompt = metrics.get("prompt_tokens")
        if prompt:
            stats["model_calls"] += 1
            stats["output_tokens"] += metrics.get("completion_tokens") or 0
            peak_prompt = max(peak_prompt, prompt)
            # Compaction replaces old events with a summary: the agent's next prompt is much smaller than its last.
            # A helper (agent_tool) starts a fresh conversation on every call, so a new call resets its baseline.
            author = (step.get("extra") or {}).get("author")
            previous = last_prompt.get(author)
            if previous and prompt < 0.7 * previous:
                stats["compactions"] += 1
            last_prompt[author] = prompt
        for call in step.get("tool_calls") or []:
            name = call.get("function_name")
            tools[name] += 1
            if name == "run_command":
                commands[(call.get("arguments") or {}).get("command", "")] += 1
            if name in HELPER_FORMATS:
                last_prompt.pop(name, None)
                helper_total[name] += 1
                reply = str((step.get("observation") or {}).get("content", ""))
                if HELPER_FORMATS[name] in reply[:200]:
                    helper_ok[name] += 1
    stats["peak_prompt"] = peak_prompt
    stats["repeated_cmds"] = sum(n - 1 for n in commands.values() if n > 1)
    helpers = {h: f"{helper_ok[h]}/{helper_total[h]}" for h in helper_total}
    return stats, tools, helpers


def analyze(results_dir):
    results_dir = Path(results_dir)
    tasks = [json.loads(line) for line in (results_dir / "task_results.jsonl").read_text().splitlines() if line]
    print(f"\n## {results_dir.name}: {sum(t.get('resolved', False) for t in tasks)}/{len(tasks)} resolved")
    header = (f"{'task':<16} {'outcome':<18} {'sec':>5} {'calls':>5} {'out_tok':>7} {'peak_in':>7} "
              f"{'compact':>7} {'nudge':>5} {'repeat':>6}  tools / helpers")
    print(header)
    print("-" * len(header))
    for task in tasks:
        task_id = task.get("instance_id") or task.get("task_id")
        trace_file = results_dir / "traces" / f"trace_{task_id}.json"
        trace = json.loads(trace_file.read_text()) if trace_file.exists() else {}
        stats, tools, helpers = trace_stats(trace)
        tool_list = ", ".join(f"{name}×{n}" for name, n in tools.most_common())
        helper_list = (" | format ok " + ", ".join(f"{h} {v}" for h, v in helpers.items())) if helpers else ""
        print(f"{task_id:<16} {bucket(task):<18} {task.get('duration_seconds', 0):>5.0f} {stats['model_calls']:>5} "
              f"{stats['output_tokens']:>7} {stats['peak_prompt']:>7} {stats['compactions']:>7} "
              f"{stats['nudges']:>5} {stats['repeated_cmds']:>6}  {tool_list}{helper_list}")
        if task.get("error"):
            print(f"{'':<16} error: {task['error'][:150]}")


for name in CONFIGS:
    analyze(RESULTS / name)
shutil.make_archive(str(WORKING_DIR / 'results'), 'zip', RESULTS)
print('\nWall clock per config (min):', {k: round(v / 60, 1) for k, v in wall.items()})
print('Wrote /kaggle/working/results.zip')